In [1]:
from Lists.abbreviations import INTERNET_SLANG
from Lists.emoticons import EMOTICONS
from Lists.emojis import EMO_UNICODE
import pandas as pd
import contractions
import unicodedata
import re

In [2]:
df = pd.read_csv('./datasets/all_datasets.csv')
df.head()

,text,type,label
0,@discerningmumin Because while other groups ca...,cyberbullying,1
1,` == New Discussion == While I understan...,not_cyberbullying,0
2,` == [[[No page]]] was reviewed by Tentinator...,not_cyberbullying,0
3,*:Didn't we have a 10 year old admin at one p...,not_cyberbullying,0
4,@MarvinJRees you shouldnt have taken the statu...,cyberbullying,1


In [4]:
UNICODE_EMO = {v: k for k, v in EMO_UNICODE.items()}


def convert_emoticons(text):
    for emot in EMOTICONS:
        text = re.sub(u'(' + emot + ')', "  ".join(EMOTICONS[emot].replace(",", "").split()), text)
    return text


def convert_emojis(text):
    for emot in UNICODE_EMO:
        text = re.sub(r'(' + emot + ')', "  ".join(UNICODE_EMO[emot].replace(",", "").replace(":", "").split()), text)
    return text


def convert_abbreviations(text):
    for abbr in INTERNET_SLANG:
        pattern = re.compile(r'\b' + re.escape(abbr) + r'\b', re.IGNORECASE)
        replacement = " ".join(INTERNET_SLANG[abbr].replace(",", "").split())
        text = pattern.sub(replacement, text)
    return text


def remove_unknown_emojis(text):
    emoji_pattern = re.compile("["
                               u"\U0001F600-\U0001F64F"  # emoticons
                               u"\U0001F300-\U0001F5FF"  # symbols & pictographs
                               u"\U0001F680-\U0001F6FF"  # transport & map symbols
                               u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
                               u"\U00002702-\U000027B0"  # miscellaneous symbols
                               u"\U0001F900-\U0001F9FF"  # supplemental symbols
                               u"\U0001F200-\U0001F251"  # enclosed characters
                               u"\U0001F004-\U0001F0CF"  # playing cards
                               u"\U00002B50"  # star symbol
                               "]+", flags=re.UNICODE)

    emojis_in_text = emoji_pattern.findall(text)

    for emoji_char in emojis_in_text:
        if emoji_char not in UNICODE_EMO:
            text = text.replace(emoji_char, '')
    return text


def remove_accented_chars(text):
    text = contractions.fix(text)
    return unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8', 'ignore')

In [5]:
def convert_emojis_and_slang(text):
    text = convert_emoticons(text)
    text = convert_emojis(text)
    text = remove_unknown_emojis(text)
    text = convert_abbreviations(text)
    text=remove_accented_chars(text)

    return text

In [6]:
from tqdm.notebook import tqdm

In [7]:
def preprocess_text(df_input):
    tqdm.pandas() 

    df_comments = df_input.copy()
    df_comments['text'] = df_comments['text'].str.replace(r"http\S+", "URL", regex=True)  # remove URLs
    df_comments['text'] = df_comments['text'].str.replace(r'@[A-Za-z0-9_.]+', 'USER', regex=True)  # remove user mentions
    df_comments['text'] = df_comments['text'].str.replace(r'#+(\S+)', r'\1', regex=True)  # remove hashtags
    df_comments['text'] = df_comments['text'].str.replace(r'\s+', ' ', regex=True)  # remove extra spaces
    df_comments['text'] = df_comments['text'].str.replace(r'\d+', '', regex=True)  # remove digits
    df_comments['text'] = df_comments['text'].str.replace(r"[^a-zA-Z0-9\s\.\,\!\?\;\:\-]", '', regex=True)


    df_comments['text'] = df_comments['text'].progress_apply(lambda x: convert_emojis_and_slang(x))

    df_comments = df_comments.dropna(how='any')
    df_comments = df_comments.drop_duplicates()

    return df_comments


In [ ]:
df = preprocess_text(df)

  0%|          | 0/202952 [00:00<?, ?it/s]

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.to_csv('./all_datasets_transformers.csv',index=False)